# Actividad 06 – Ítem B: YOLO en **imágenes**
**Visión Artificial – UNA Puno – X ciclo, grupo B**

Detección con YOLO11 (ultralytics, preentrenado en COCO) sobre COCO128. Se mide **mAP** y **latencia** por imagen con **el mismo evaluador (torchmetrics)** que el ítem A, para que la comparación sea justa.

**Parámetros comunes del equipo (deben ser idénticos en A, B, C y D):** `CONF=0.25`, `IMGSZ=640`, mismos videos e imágenes (COCO128).

> Usa GPU: *Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*.

## 1. Instalación

In [ ]:
!pip -q install ultralytics torchmetrics pycocotools

## 2. Modelo YOLO

In [ ]:
import time, cv2, numpy as np

import torch
from ultralytics import YOLO

MODELO = "yolo11n.pt"   # o yolo11s.pt / yolo11m.pt
CONF  = 0.25
IMGSZ = 640
device = 0 if torch.cuda.is_available() else "cpu"
print("Dispositivo:", torch.cuda.get_device_name(0) if device == 0 else "CPU")

model = YOLO(MODELO)
nombres = [model.names[i] for i in range(len(model.names))]

def detectar_yolo(img_bgr):
    """Devuelve boxes xyxy, scores, labels (0-79) y latencia extremo a extremo en segundos."""
    t0 = time.perf_counter()
    r = model.predict(img_bgr, conf=0.001, imgsz=IMGSZ, max_det=100, device=device, verbose=False)[0]
    if device == 0: torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    b = r.boxes
    return (b.xyxy.cpu().numpy(), b.conf.cpu().numpy(),
            b.cls.cpu().numpy().astype(np.int64), dt)

## 3. Datos (COCO128 con etiquetas)

In [ ]:
import os, glob, cv2, numpy as np
!wget -q -nc https://github.com/ultralytics/assets/releases/download/v0.0.0/coco128.zip
!unzip -q -n coco128.zip

IMG_DIR = "coco128/images/train2017"
LBL_DIR = "coco128/labels/train2017"
imagenes = sorted(glob.glob(f"{IMG_DIR}/*.jpg"))
print(len(imagenes), "imágenes")

def cargar_gt(ruta_img):
    """Lee etiquetas YOLO (cls cx cy w h normalizados) y devuelve cajas xyxy en píxeles."""
    h, w = cv2.imread(ruta_img).shape[:2]
    lbl = os.path.join(LBL_DIR, os.path.basename(ruta_img).replace(".jpg", ".txt"))
    boxes, labels = [], []
    if os.path.exists(lbl):
        for line in open(lbl):
            c, cx, cy, bw, bh = map(float, line.split())
            boxes.append([(cx-bw/2)*w, (cy-bh/2)*h, (cx+bw/2)*w, (cy+bh/2)*h])
            labels.append(int(c))
    return np.array(boxes, dtype=np.float32).reshape(-1, 4), np.array(labels, dtype=np.int64)

## 4. Evaluación: mAP + latencia

In [ ]:
import time, json, torch, pandas as pd
from torchmetrics.detection import MeanAveragePrecision

def evaluar_imagenes(detectar_fn, etiqueta):
    """detectar_fn(img_bgr) -> (boxes_xyxy[N,4], scores[N], labels_coco80[N], latencia_s)
    Se corre con umbral bajo (0.001) para el mAP; CONF se aplica solo a conteos/confianza."""
    metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")
    detectar_fn(cv2.imread(imagenes[0]))   # warm-up
    filas = []
    for ruta in imagenes:
        img = cv2.imread(ruta)
        b, s, l, dt = detectar_fn(img)
        gb, gl = cargar_gt(ruta)
        metric.update(
            [dict(boxes=torch.tensor(b).reshape(-1, 4), scores=torch.tensor(s), labels=torch.tensor(l))],
            [dict(boxes=torch.tensor(gb), labels=torch.tensor(gl))])
        m = s >= CONF
        filas.append({"imagen": os.path.basename(ruta), "latencia_ms": dt*1000, "fps": 1/dt,
                      "n_detecciones": int(m.sum()),
                      "conf_media": float(s[m].mean()) if m.any() else np.nan})
    df = pd.DataFrame(filas)
    r = metric.compute()
    resumen = {"modelo": etiqueta, "imgsz": IMGSZ, "conf": CONF, "dispositivo": str(device),
               "n_imagenes": len(df),
               "mAP50": round(float(r["map_50"]), 4), "mAP50_95": round(float(r["map"]), 4),
               "mAR100": round(float(r["mar_100"]), 4),
               "latencia_media_ms": round(df.latencia_ms.mean(), 2),
               "latencia_p95_ms": round(df.latencia_ms.quantile(0.95), 2),
               "FPS_equivalente": round(1000/df.latencia_ms.mean(), 2),
               "detecciones_por_imagen": round(df.n_detecciones.mean(), 2),
               "confianza_media": round(df.conf_media.mean(), 3)}
    return df, resumen

In [ ]:
df, resumen = evaluar_imagenes(detectar_yolo, MODELO)
print(pd.Series(resumen).to_string())

## 5. Ejemplos visuales

In [ ]:
DETECTAR, NOMBRES = detectar_yolo, nombres

import matplotlib.pyplot as plt
def dibujar(img, b, s, l, nombres, conf):
    img = img.copy()
    for (x1, y1, x2, y2), sc, c in zip(b, s, l):
        if sc < conf: continue
        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        cv2.putText(img, f"{nombres[c]} {sc:.2f}", (int(x1), max(15, int(y1)-5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    return img

fig, ax = plt.subplots(1, 4, figsize=(20, 5))
for a, ruta in zip(ax, imagenes[:4]):
    im = cv2.imread(ruta); b, s, l, _ = DETECTAR(im)
    a.imshow(cv2.cvtColor(dibujar(im, b, s, l, NOMBRES, CONF), cv2.COLOR_BGR2RGB)); a.axis("off")
plt.tight_layout(); plt.savefig("ejemplos_yolo.png", dpi=120); plt.show()

## 6. Guardar y descargar

In [ ]:
df.to_csv("metricas_imagenes_yolo.csv", index=False)
json.dump(resumen, open("resumen_imagenes_yolo.json", "w"), indent=1)
from google.colab import files
for f in ["metricas_imagenes_yolo.csv", "resumen_imagenes_yolo.json", "ejemplos_yolo.png"]:
    files.download(f)